# Time-Series Forecasting — Walk-Forward CV, Lag Features, and Baselines

<hr>

<center>
<div>
<img src="https://raw.githubusercontent.com/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/main/notebooks/figures/mgmt_474_ai_logo_02-modified.png" width="200"/>
</div>
</center>

# <center><a class="tocSkip"></center>
# <center>QM47400 Predictive Analytics</center>
# <center>Professor: Davi Moreira </center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/blob/main/notebooks/nb16_time_series_forecasting_student.ipynb)


> **📋 Participation Reminder:** This notebook contains **2 PAUSE-AND-DO exercises**. Complete both to receive participation credit.

---


## Learning Objectives

By the end of this notebook, you will be able to:

1. Distinguish a forecasting problem from a generic supervised-learning problem and choose the right evaluation protocol.
2. Run a structured time-series EDA — time plot, seasonal sub-series, decomposition, autocorrelation — on a real labor-market dataset.
3. Build a **time-respecting** 80/20 train/test split where the test window is the most recent slice of history, and walk-forward CV replaces the separate validation set.
4. Run **walk-forward cross-validation** with `ExpandingWindowSplitter` from `sktime` instead of k-fold CV (which would shuffle time and leak the future into the past).
5. Compare four classical forecasting benchmarks (Mean, Naive, Seasonal-Naive, Drift) against a learned **lag-feature linear regression** on identical CV folds.
6. Add **regularization** (Ridge) to the lag-feature linear model and decide whether it earns its place via the Student's *t* 95% CI overlap rule.
7. Open the locked test window in a **one-shot evaluation ceremony**, mirroring nb14's protocol but adapted to time.


## Setup

Import the libraries we will use across the notebook. Most of the toolkit is familiar from Week 1 — `pandas`, `matplotlib`, `seaborn`, `LinearRegression`, `Ridge`, `mean_absolute_error`, and `scipy.stats.t` for the Student's *t* CIs from nb08. The EDA tools are the same: `STL` and `plot_acf` from `statsmodels`.

What is new today is **`sktime`** — the time-series forecasting framework introduced in the lecture slides. Three `sktime` tools replace the manual code you would otherwise have to write by hand: **`NaiveForecaster`** (the four classical benchmarks in one class), **`make_reduction`** (wraps any sklearn `Pipeline` as a time-respecting forecaster with automatic lag features — the nb02 Pipeline principle applied to forecasting), and **`ExpandingWindowSplitter`** (walk-forward CV that replaces `TimeSeriesSplit`). The `sktime` API mirrors sklearn's `.fit()` / `.predict()` interface, so the vocabulary is familiar — only the time-respecting plumbing underneath is new.

In [ ]:
# Setup Cell — install sktime (not pre-installed in Colab)
!pip install -q sktime

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from statsmodels.tsa.seasonal import STL
from statsmodels.graphics.tsaplots import plot_acf

# sktime — time-series forecasting framework (lecture 08 vocabulary)
from sktime.forecasting.naive import NaiveForecaster
from sktime.forecasting.compose import make_reduction
from sktime.split import temporal_train_test_split, ExpandingWindowSplitter

# sklearn — the estimators that sktime wraps via make_reduction
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.metrics import mean_absolute_error
from scipy.stats import t as student_t

warnings.filterwarnings("ignore")

RANDOM_SEED = 474
np.random.seed(RANDOM_SEED)
plt.rcParams['figure.figsize'] = (10, 6)
pd.set_option('display.precision', 3)
sns.set_style("whitegrid")

print("Setup complete!")
print(f"RANDOM_SEED = {RANDOM_SEED}")

**Reading the output:** A clean `Setup complete!` confirms all libraries loaded without errors. The `!pip install -q sktime` line installs `sktime` in Colab (it is not pre-installed). If the install fails, try `Runtime → Restart runtime` and re-run the cell. The `RANDOM_SEED = 474` and `figure.figsize = (10, 6)` settings match every prior notebook, so your outputs will reproduce exactly.

---

## 1. Why This Matters: Forecasting US Retail Employment

The **US Bureau of Labor Statistics** publishes monthly employment counts for every major industry. A **state-level workforce planner** uses those numbers to forecast next year's retail labor demand:

> *"I need a defensible forecast of US retail-sector employment one year out — with a confidence interval the legislature can read. Last year's headline number is not enough; I need the model and the diagnostics."*

This is **not** the kind of problem we solved in nb01–nb15. There, every row was an independent observation and a 60/20/20 random split was the right protocol. Here, the rows are months in a sequence — the **order matters**, and shuffling them would let the model peek at the future during training (a classic data leak). The fix is structural: the test window is always the **most recent** slice of history, and cross-validation walks forward in time.

This notebook ports the **Week-1 analytics workflow** (EDA → split → baselines → linear features → regularization) to the time-series setting, threading the same structural rule throughout: *the future cannot leak into the past.*

**A question that often comes up here:** *"Why isn't this just nb14 with a different metric?"* Two reasons. First, k-fold CV with shuffled rows would let row 50 be in the training fold and row 49 in the validation fold — the model would see a future month while learning to predict an earlier one. That defeats the entire idea of forecasting. Second, employment series have **seasonality** (holiday hiring) and **long-run trend** (decades of structural growth), so a feature engineered as "value 12 months ago" is structurally meaningful in a way that "row 12 in the dataset" is not.


## 2. Load the Data and Sanity Checks

We use the **US Employment dataset** from *Forecasting: Principles and Practice* (FPP3), the standard open-source textbook for time-series analysis. The dataset contains monthly employment counts (in thousands) for every major BLS industry classification from 1939 onward — roughly 80 years of monthly observations across dozens of industries. We filter to the **"Retail Trade"** series because it is the workforce planner's target and because it has all three structural features that make forecasting interesting: long-run trend, clear annual seasonality, and occasional recession-driven disruptions. The `ds` column is the date stamp; `y` is the employment count.

In [ ]:
# Load the full dataset directly from the course's GitHub raw URL
DATA_URL = (
    "https://raw.githubusercontent.com/davi-moreira/"
    "2026Summer_predictive_analytics_purdue_MGMT474/main/"
    "lecture_slides/08_time_series/data/us_employment.csv"
)
us_employment = pd.read_csv(DATA_URL, parse_dates=["ds"])

# Filter to the Retail Trade series
df = (
    us_employment.query('unique_id == "Retail Trade"')
    .loc[:, ["ds", "y"]]
    .sort_values("ds")
    .reset_index(drop=True)
)

# Convert to sktime-compatible format: a pandas Series with a PeriodIndex.
# sktime forecasters expect a time-indexed Series, not a DataFrame.
y = df.set_index("ds")["y"]
y.index = y.index.to_period("M")

print(f"Rows: {len(y):,}")
print(f"Date range: {y.index[0]}  ->  {y.index[-1]}")
print(f"Missing values: {y.isna().sum()}")
print()
df.head()

**Reading the output:**

You should see roughly **960 rows** spanning **1939-01 through 2019-09** (80 years of monthly data) with **zero missing values**. The two columns are `ds` (date stamp) and `y` (employment in thousands). After loading, we convert the DataFrame to a **pandas Series with a PeriodIndex** (`y.index.to_period("M")`) — this is the format `sktime` forecasters expect. The DataFrame `df` stays around for the EDA plots in §3, which use `df["ds"]` and `df["y"]` directly.

> **A question that often comes up here:** *"Why is `unique_id` a string column?"* The original dataset is in long format — one row per (industry × month). The `unique_id` column tags each row with its industry. We filter to a single one so the analysis stays focused.

## 3. Time-Series EDA — Six Plots, One Story

A time series asks for visual EDA before any modeling. The canonical sequence is: **time plot** (the whole series), **time plot zoomed** (a recent slice), **seasonal sub-series** (per-month box plot), **STL decomposition** (trend + seasonal + remainder), **ACF** (autocorrelation function), **lag-1 scatter** (do consecutive months track each other?). Six plots, one combined story.


### 3.1 Time plot — the whole series

The first plot every forecaster makes. Plot the full series on a timeline and let the shape speak: an upward or downward drift means **trend**, regular ripples mean **seasonality**, and sharp discontinuities mean **structural breaks** (recessions, policy changes). No statistical test conveys these features as quickly as a single well-drawn time plot — and the workforce planner who skips this step risks building a model that ignores structure the eye catches in seconds.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df["ds"], df["y"], color="#1f77b4", linewidth=0.8)
ax.set_xlabel("Year")
ax.set_ylabel("Employment (thousands)")
ax.set_title("US Retail Trade Employment, 1939–2019 (monthly)")
plt.tight_layout()
plt.show()


**Reading the output:**

Three structural components are visible in this single plot, and the workforce planner needs to name all three before modeling begins.

**1. Long-run trend.** Employment roughly **triples** from about 5,000 thousand in 1939 to over 15,000 thousand by 2019. The growth is not uniform — you can trace the mid-century post-war expansion, the 1990s retail and e-commerce boom, and the gradual recovery after 2010. The overall shape is an upward curve that any forecasting model must track; a model that ignores it would predict the 1950s level for 2020.

**2. Seasonality.** Look closely and you will see small annual ripples running along the trend line — the curve is not smooth but gently serrated. Those ripples are **holiday retail hiring** in November and December (the peaks) and **post-holiday layoffs** in January and February (the troughs). At this 80-year zoom level the ripples are hard to read, which is exactly why the next plot zooms in. But even here, the regularity is visible: the same up-down pattern repeats every 12 months for eight decades. A model that captures trend but ignores seasonality will systematically over-predict in January and under-predict in December.

**3. Structural breaks.** The sharpest disruptions are **recessions**: the 2008–2010 financial crisis is the most dramatic (a steep drop of roughly 2,000 thousand employees followed by a multi-year recovery), but smaller dips are visible in 1974 (oil crisis), 1980 (double-dip recession), 1990, and 2001 (dot-com bust). These breaks are driven by macroeconomic shocks that the series itself cannot predict — no lag feature or seasonal pattern will warn you that a financial crisis is coming. The practical implication for the workforce planner: the model's prediction interval must be wide enough to cover these tail events, and any forecast delivered to the legislature should carry a caveat about recession-driven uncertainty.

A single forecasting model has to capture trend and seasonality (the learnable components) while honestly acknowledging that structural breaks will occasionally push actuals outside even a well-calibrated prediction interval.

### 3.2 Time plot zoomed — last 10 years

The full 80-year time plot compresses 960 monthly observations into a single curve, which makes the annual seasonal cycle nearly invisible — the ripples are too small relative to the 80-year trend. Zooming in to the last decade stretches the x-axis enough to see individual holiday peaks and post-holiday dips. This is the plot that tells the workforce planner *how much* staffing swings within a single year.

In [ ]:
recent = df[df["ds"] >= "2010-01-01"]
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(recent["ds"], recent["y"], "o-", color="#2ca02c", markersize=4)
ax.set_title("US Retail Trade Employment — 2010–2019 (monthly)")
ax.set_xlabel("Year")
ax.set_ylabel("Employment (thousands)")
plt.tight_layout()
plt.show()


**Reading the output:**

The zoomed plot makes the seasonal cycle unmistakable. Three features are now readable that the 80-year plot compressed into illegibility.

**Annual peaks in November/December.** Every year, employment surges as retailers staff up for the holiday season — Black Friday through Christmas. The peaks are the tallest points in each annual wave. For the workforce planner, these peaks are not surprises; they are predictable staffing events that drive temporary-hire budgets, training timelines, and warehouse capacity decisions months in advance.

**Post-holiday troughs in January/February.** After the holidays, seasonal positions end and employment drops sharply. The trough is typically the lowest point of the annual cycle. The gap between the December peak and the January trough — a swing of several percent of the total retail workforce — is the seasonal amplitude the model needs to capture.

**Slow upward trend underneath the waves.** Each year's trough is slightly higher than the previous year's trough; each peak is slightly higher than the previous peak. That is the long-run trend visible at this scale — the same structural growth that §3.1's full time plot showed over 80 years, now legible as a gentle upward tilt beneath the seasonal ripples.

This seasonal cycle is exactly the structure that a **lag-12 feature** (this month vs. the same month last year) will capture automatically in section 8. The model does not need to "know" about holiday hiring — it just needs to see that December 2018 looked a lot like December 2017.

### 3.3 Seasonal sub-series box plot

A box plot of `y` grouped by **month-of-year** compresses 80 Decembers into one box, 80 Januaries into another, and so on. It answers two questions at once: *"how strong is the seasonal pattern?"* (do the medians differ across months?) and *"how stable is it?"* (are the boxes tight or sprawling?). If December's box sits consistently above June's across 80 years, the seasonal effect is real and worth modeling. If every month's box overlaps every other, there is no seasonality to capture.

In [ ]:
df_seasonal = df.copy()
df_seasonal["month"] = df_seasonal["ds"].dt.month
fig, ax = plt.subplots(figsize=(11, 5))
sns.boxplot(data=df_seasonal, x="month", y="y", ax=ax, color="#ff7f0e")
ax.set_title("Seasonal sub-series — employment distribution by month-of-year")
ax.set_xlabel("Month of year")
ax.set_ylabel("Employment (thousands)")
plt.tight_layout()
plt.show()


**Reading the output:**

The box plot answers both seasonal questions at once.

**How strong is the seasonal pattern?** Strong enough to act on. **December** sits visibly above every other month — this is the holiday retail hiring surge at its clearest. November comes next (early holiday ramp-up and pre-Black Friday staffing), then a steady plateau from March through October, then the **January/February dip** as seasonal positions end and post-holiday returns wind down. The medians trace a smooth annual cycle that the workforce planner can use as a staffing calendar: expect the highest labor demand in December, the lowest in January, and a stable mid-range from spring through early fall.

**How stable is the seasonal pattern?** Stable in shape, but the boxes are wide. That width is not noise — it is the **long-run trend hiding inside the by-month aggregation**. Think about what a single "December" box contains: December 1945 (roughly 6,000 thousand employees) and December 2015 (roughly 16,000 thousand). Both are Decembers, but they sit at very different absolute levels because the economy grew over those 70 years. When you lump them into one box, the box stretches from 6,000 to 16,000 — a range driven by the trend, not by December-to-December instability. The seasonal *shape* (December > November > ... > January) is real and consistent across decades; the seasonal *amplitude* relative to the trend is moderate.

This distinction — real seasonal shape but trend-inflated box width — is why the STL decomposition in the next plot is valuable. STL separates the trend from the seasonal component mathematically, so you can see each one's contribution cleanly.

### 3.4 STL decomposition — trend + seasonal + remainder

**STL (Seasonal-Trend decomposition using Loess)** splits the series into three additive components:

$$y_t = T_t + S_t + R_t$$

Here is what each term means in plain language:

- $y_t$ is the **actual observed value** — the employment count in month $t$ (the raw data you have been plotting).
- $T_t$ is the **trend** — the smooth, long-run trajectory of the series after stripping away the seasonal ups and downs. Think of it as the answer to *"ignoring the holiday cycle, where is employment heading?"*
- $S_t$ is the **seasonal component** — the regular, repeating annual pattern (December peaks, January troughs) that cycles every 12 months. It captures the *predictable calendar-driven swings* the workforce planner staffs around.
- $R_t$ is the **remainder** (also called the residual) — everything the trend and seasonal components cannot explain. Small remainder values mean the decomposition captured most of the structure; large spikes (like around the 2008 recession) mean something happened that neither the trend nor the seasonal pattern anticipated.

The equation says: at any month $t$, the actual value is the sum of these three pieces. It is the time-series analog of "explain the variance and look at the residuals" — the trend and seasonal components are the explained part; the remainder is what is left over.

In [ ]:
# STL decomposition with monthly period
ts = df.set_index("ds")["y"]
stl = STL(ts, period=12, robust=True).fit()

fig, axes = plt.subplots(4, 1, figsize=(12, 9), sharex=True)
axes[0].plot(ts.index, ts.values, color="black"); axes[0].set_ylabel("y (data)")
axes[1].plot(ts.index, stl.trend, color="#1f77b4"); axes[1].set_ylabel("Trend")
axes[2].plot(ts.index, stl.seasonal, color="#2ca02c"); axes[2].set_ylabel("Seasonal")
axes[3].plot(ts.index, stl.resid, color="#d62728"); axes[3].set_ylabel("Remainder")
axes[3].axhline(0, color="black", linewidth=0.5)
axes[0].set_title("STL Decomposition — US Retail Trade Employment")
plt.tight_layout()
plt.show()


**Reading the output:**

Four panels, top to bottom, each isolating one component of the series.

**Data (top panel).** The raw series — the same curve you saw in §3.1's time plot. It contains all three structural components mixed together.

**Trend (second panel).** The smooth long-run component after the seasonal and residual fluctuations have been stripped away. You can now see the structural growth trajectory clearly: steady expansion from the 1940s through the 1970s, a plateau and mild dips around the oil-crisis recessions, accelerating growth through the 1990s retail boom, the sharp **2008 financial-crisis drop** (roughly 2,000 thousand employees lost in two years), and the gradual post-2010 recovery. This is the component that lag-1 features will capture — each month's employment level is closely related to last month's.

**Seasonal (third panel).** The regular annual pattern, isolated from the trend. The same wave shape repeats every 12 months for 80 years — December peaks, January troughs, a consistent amplitude throughout. Notice that the wave height does not grow over time even though the trend does; this confirms the **additive** decomposition was the right choice. If the seasonal swings had grown proportionally with the level (bigger absolute swings at higher employment), a multiplicative decomposition would have been needed instead.

**Remainder (bottom panel).** What neither trend nor seasonality explains. Most of the time, the remainder fluctuates in a narrow band around zero — the trend and seasonal components account for nearly all of the variation. But there are visible **spikes around recessions**: the 2008 crisis produces the largest residual, and smaller spikes appear in 1974, 1980, 1990, and 2001. These are the structural breaks from §3.1 — macroeconomic shocks that arrive from outside the series. No lag feature or seasonal pattern will predict them; the prediction interval in §9 will need to be wide enough to accommodate them.

> **A question that often comes up here:** *"Should I use additive or multiplicative decomposition?"* Additive when the seasonal amplitude does not grow with the trend; multiplicative when it does. Visually, the seasonal swing here is roughly the same height in 1950 (small absolute employment) as in 2010 (large absolute employment) → additive is the right call. If the seasonal swing was larger in absolute terms when employment was higher, multiplicative would fit better.

### 3.5 Autocorrelation (ACF) plot

The ACF asks *"how strongly does month $t$'s value depend on month $t-k$'s value, for each lag $k$?"*. Formally, the autocorrelation at lag $k$ is:

$$r_k = \frac{\sum_{t=k+1}^{n}(y_t - \bar{y})(y_{t-k} - \bar{y})}{\sum_{t=1}^{n}(y_t - \bar{y})^2}$$

Here is what each piece means:

- $y_t$ is the employment value in month $t$ (the current month you are looking at).
- $y_{t-k}$ is the employment value $k$ months earlier — the "lagged" value. When $k = 1$, it is last month; when $k = 12$, it is the same month one year ago.
- $\bar{y}$ is the overall mean of the series — the average employment across all 960 months.
- The numerator measures how much month $t$ and month $t-k$ move together *relative to the mean*. If both are above average at the same time (or both below), the product is positive and the correlation is strong.
- The denominator is the total variance of the series — it scales the numerator so $r_k$ always falls between $-1$ and $+1$.

In practice, you do not compute this by hand — `plot_acf` does it for you. What matters is reading the bar chart: each bar is one $r_k$ value. Tall bars at lags 1, 2, 3 mean **trend and momentum** — recent months are highly correlated with the present. A tall bar at lag 12 (and again at 24) means **annual seasonality** — what happened a year ago is a strong predictor of what happens now. The ACF is the empirical tool that tells us *which lags are worth engineering as features*.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
plot_acf(ts.values, lags=36, ax=ax, zero=False)
ax.set_title("Autocorrelation function — lags 1 through 36")
ax.set_xlabel("Lag (months)")
plt.tight_layout()
plt.show()


**Reading the output:**

The ACF translates the visual patterns from the first four plots into precise numerical evidence about which lags carry predictive signal.

**Slow decay across lags 1–24.** The bars start tall at lag 1 (correlation above 0.95) and decline gradually through lag 24. This slow decay is the **signature of a strong trend** — when employment is high this month, it was also high last month, and the month before that, and so on. The practical implication: the most recent value (`lag1`) is by far the single most informative predictor of the next value. This is why the naive forecast ("next month = last month") will be so hard to beat.

**Local peaks at lags 12 and 24.** On top of the slow decay, the bars at lags 12 and 24 are visibly taller than their neighbors. Lag 12 means "this month correlates strongly with the same month one year ago" — that is the annual seasonal cycle the zoomed time plot and the box plot already showed. Lag 24 means the same pattern holds two years back. These peaks are weaker than lag 1 (the trend dominates), but they carry **independent seasonal information** that lag 1 alone cannot provide.

**The blue shaded band** marks the 95% confidence interval for "no significant autocorrelation." Every bar that extends beyond the band is statistically significant. On this series, every bar through lag 36 is significant — the series has strong, persistent structure at every timescale up to three years.

This ACF gives us **direct empirical justification** for the two lag features we will engineer in section 8: `lag1` captures the trend and short-term momentum (the slow decay), and `lag12` captures the annual seasonal cycle (the lag-12 peak). The ACF is the diagnostic; the features are the response.

### 3.6 Lag-1 scatter

The simplest forecast in the world is *"next month equals last month."* The lag-1 scatter plots this month's employment against last month's — every dot is one month. If the dots hug the 45° line, the naive forecast is strong (month-to-month changes are small). If they scatter into a cloud, consecutive months are volatile and the naive forecast will have large errors. For the workforce planner, this plot answers a practical question: *"can I get away with just using last month's number, or do I genuinely need a model?"*

In [ ]:
lag1_df = df.assign(lag1=df["y"].shift(1)).dropna()
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(lag1_df["lag1"], lag1_df["y"], s=8, alpha=0.5, color="#9467bd")
lo, hi = lag1_df["y"].min(), lag1_df["y"].max()
ax.plot([lo, hi], [lo, hi], "k--", linewidth=0.8, label="Perfect lag-1 forecast (y = lag1)")
ax.set_xlabel("Employment, t-1 (lag1)")
ax.set_ylabel("Employment, t")
ax.set_title("Lag-1 scatter — does the previous month predict the current month?")
ax.legend()
plt.tight_layout()
plt.show()


**Reading the output:**

Every dot in this scatter is one month. The x-coordinate is last month's employment; the y-coordinate is this month's. The dashed diagonal is the 45° line — the line where "this month = last month" exactly.

**The dots hug the 45° line tightly.** Month-to-month changes in retail employment are small relative to the overall level. A month at 15,000 thousand employees is almost always followed by a month between 14,800 and 15,200 — a change of at most 1–2%. That tightness is the visual proof that the **naive forecast** ("next month = last month") will be a strong baseline. Any learned model that does not beat it is not earning its keep.

**The cloud is elongated, not round.** The scatter stretches from the lower-left (early decades, lower employment) to the upper-right (recent decades, higher employment). That elongation is the trend — the series moves through different employment regimes over 80 years. Within each regime, the dots cluster tightly around the diagonal.

**There are no dramatic outliers far from the line.** Even the recession months (2008–2009) do not produce dots that land far off the diagonal, because the employment drops happened over multiple months rather than in a single catastrophic jump. This is good news for the lag-feature regression: the relationship between consecutive months is approximately linear and stable.

> **A question that often comes up here:** *"Does this mean the naive forecast will always be hard to beat?"* For slow-moving, strongly autocorrelated series like monthly employment, yes — the naive forecast inherits most of the signal for free. But for volatile series — daily tech stocks, hourly web traffic, cryptocurrency — the lag-1 scatter would show a much wider cloud, and the naive baseline would have much larger errors. The tighter the scatter hugs the 45° line, the higher the bar any learned model has to clear.

Six plots, one combined story: strong trend, clear annual seasonality, and tight lag-1 autocorrelation. Section 4 translates those structural facts into the single rule that governs every modeling decision below.

---

## 4. The Structural Rule — Never Shuffle, Never Leak the Future

Every static-classification rule we built since nb01 still works in this notebook — **except one**. Rows here are months in a sequence, and shuffling them would let the model peek at the future during training. That single structural change cascades into three downstream changes:

1. **Train/test split**: the test window is the **most recent slice** of history, not a random sample.
2. **Cross-validation**: every fold's training data must come strictly **before** its validation data.
3. **Features**: lag features (last month, 12 months ago) replace random feature engineering.

Sections 5–7 implement each one in order.

---


## 5. Time-Respecting 80/20 Split — CV Replaces the Validation Set

In nb01–nb14, we used a 60/20/20 split: 60% train, 20% validation for single-split evaluation, 20% locked test. The validation set gave us a quick performance estimate before the test ceremony. But starting in nb08, **cross-validation replaced single-split evaluation** as the primary model-selection tool — the CV mean and 95% CI are more reliable than any single validation score.

For forecasting, there is an additional reason to skip the separate validation window: **the most recent history is the most valuable for predicting the future.** A monthly employment series has trend and seasonality that evolve over decades. Holding out 20% of the history as a validation set means the model never trains on the most recent pre-test data — exactly the data that best reflects current dynamics. Walk-forward CV on the full pre-test window gives us honest model comparison *and* lets the model see the most recent patterns during each fold's training.

So today we use an **80/20 split**: the oldest 80% is the **training window** (used for CV-based model selection), and the most recent 20% is the **locked test window** (touched once in §9). `temporal_train_test_split` enforces the chronological ordering.

In [ ]:
# Time-respecting 80/20 split: oldest 80% for training (CV-based
# model selection), most recent 20% locked for the one-shot test ceremony.
y_train, y_test = temporal_train_test_split(y, test_size=0.20)

print(f"Train: {y_train.index[0]} -> {y_train.index[-1]}  (n={len(y_train)})")
print(f"Test : {y_test.index[0]} -> {y_test.index[-1]}  (n={len(y_test)})  [LOCKED]")

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(y_train.index.to_timestamp(), y_train.values, color="#1f77b4", label="Train (80%)")
ax.plot(y_test.index.to_timestamp(), y_test.values, color="#d62728",
        linestyle="--", label="Test (20%, locked)")
ax.axvline(y_train.index[-1].to_timestamp(), color="grey", linestyle=":", alpha=0.7)
ax.set_title("Time-Respecting 80/20 Split — CV replaces the validation set")
ax.legend()
plt.tight_layout()
plt.show()

**Reading the output:**

The plot shows two chronological segments separated by a vertical grey dashed line.

**Blue (Train, 80%)** covers roughly 1939 through the early 2000s — about 775 months. This is the data used for all model fitting and all cross-validation. Walk-forward CV (§6) will split this window internally into expanding training folds and validation folds, so we get honest model comparison without needing a separate held-out validation set.

**Dashed red (Test, 20%, LOCKED)** covers the most recent portion — roughly the early 2000s through 2019, about 194 months. This window includes the 2008 recession and the post-crisis recovery. It stays sealed until the §9 ceremony.

> **A question that often comes up here:** *"Why 80/20 instead of the 60/20/20 from nb01?"* In nb01–nb07, a separate validation set was the only way to evaluate models before touching the test set. From nb08 onward, cross-validation replaced that role — it gives a mean and a 95% CI from multiple folds, which is more reliable than any single validation score. For forecasting, there is a second reason: the most recent pre-test data is the most informative for predicting the future. Reserving 20% of the history as a validation window would hide the most current dynamics from the model. By combining train and val into one 80% window, every CV fold can potentially train on the most recent available data.

The split defines *which* rows go where; section 6 defines *how* to evaluate models honestly on the training portion using walk-forward cross-validation.

---

## 6. Walk-Forward Cross-Validation with `ExpandingWindowSplitter`

In nb08, `KFold` shuffled rows into training and validation folds — fine when rows are independent. Here, shuffling would let the model see future months while training on earlier ones. `sktime`'s **`ExpandingWindowSplitter`** is the structural fix. It makes three things explicit that sklearn's `TimeSeriesSplit` hides: the **initial training window** (how many months fold 1 trains on), the **step length** (how far the window advances per fold), and the **forecast horizon** (`fh` — how many months ahead each fold predicts). The training window grows with each fold, mimicking real-world deployment where you retrain monthly on an ever-growing history.

In [ ]:
# Walk-forward CV with sktime's ExpandingWindowSplitter.
# Three explicit parameters (vs TimeSeriesSplit's implicit defaults):
#   initial_window — how many months the first fold trains on
#   step_length    — how many months the window advances per fold
#   fh             — the forecast horizon (how far ahead each fold predicts)
cv = ExpandingWindowSplitter(
    initial_window=int(len(y_train) * 0.5),
    step_length=int(len(y_train) * 0.1),
    fh=np.arange(1, 13),
)

n_folds = cv.get_n_splits(y_train)

# Fold calendar: train and val date ranges per fold
print(f"Walk-forward CV: {n_folds} folds\n")
print(f"{'Fold':<6} {'Train start':<14} {'Train end':<14} {'Val start':<14} {'Val end':<14} {'Train n':<10} {'Val n'}")
print("-" * 90)
for i, (tr_idx, va_idx) in enumerate(cv.split(y_train)):
    tr_start, tr_end = y_train.index[tr_idx[0]], y_train.index[tr_idx[-1]]
    va_start, va_end = y_train.index[va_idx[0]], y_train.index[va_idx[-1]]
    print(f"{i+1:<6} {str(tr_start):<14} {str(tr_end):<14} {str(va_start):<14} {str(va_end):<14} {len(tr_idx):<10} {len(va_idx)}")

# Visualization
fig, ax = plt.subplots(figsize=(11, 4))
for fold, (tr_idx, va_idx) in enumerate(cv.split(y_train)):
    ax.plot(tr_idx, [fold]*len(tr_idx), "s", color="#1f77b4", markersize=3,
            label="train" if fold == 0 else "")
    ax.plot(va_idx, [fold]*len(va_idx), "s", color="#ff7f0e", markersize=3,
            label="val" if fold == 0 else "")
ax.set_yticks(range(n_folds))
ax.set_yticklabels([f"fold {i+1}" for i in range(n_folds)])
ax.set_xlabel("Month index in training data")
ax.set_title("Walk-Forward CV: train (blue) always precedes val (orange)")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

**Reading the output:**

The visualization shows five rows — one per CV fold — with blue squares marking training months and orange squares marking validation months.

**Fold 1** (top row) has the smallest training window and the first validation window. The model sees only the earliest portion of the training data and is tested on the months that immediately follow. This is the hardest fold — the model has the least history to learn from.

**Fold 5** (bottom row) has the largest training window — roughly five times as much data as fold 1 — and the last validation window. This is the easiest fold and the one closest to what deployment looks like: the model has nearly all the available training history.

Two structural facts to carry forward. First, **orange always sits to the right of blue** — the model never sees a future month while learning to predict an earlier one. That is the time-respecting constraint that makes walk-forward CV honest. Second, **the training window grows** across folds. That growth is not a bug; it mimics real-world deployment, where you retrain monthly on an ever-growing history and always forecast into unseen time. The consequence is that earlier folds are harder and later folds are easier — which is why the per-fold MAEs in §9 will show some variation.

> **A question that often comes up here:** *"Does the growing window give later folds an unfair advantage?"* Yes, and that is realistic. In deployment, you always have more history than you did six months ago. The expanding-window design captures that asymmetry honestly. A fixed-size sliding window (where you drop the oldest rows as you add new ones) is an alternative when you believe only recent history is relevant — but for a series with an 80-year trend, throwing away the early decades would discard useful signal.

With the walk-forward folds in hand, the next question is what to measure on each fold. Section 7 introduces four forecasting metrics and runs the classical benchmarks under all of them.

---

## 7. Forecasting Metrics — Four Benchmarks Under Four Lenses

Before evaluating any model, decide what *good* means. Four metrics show up in every time-series textbook; each answers a slightly different question about forecast quality. Sub-section 7.1 defines each metric with its formula; 7.2 gives a quick "when to use which" guide; the code cell that follows runs the four classical benchmarks under all four metrics on the validation window.


### 7.1 The four metrics

Each metric compares the actual values ($y_t$) against the forecasted values ($\hat{y}_t$) across $n$ time periods. In every formula below, $y_t$ is "what actually happened in month $t$" and $\hat{y}_t$ is "what the model predicted for month $t$."

**Mean Absolute Error (MAE).**

$$\text{MAE} = \frac{1}{n}\sum_{t=1}^{n} \left| y_t - \hat{y}_t \right|$$

The difference $y_t - \hat{y}_t$ is the forecast error for month $t$ — positive when the model under-predicted, negative when it over-predicted. The absolute value $|\ |$ strips the sign so over-predictions and under-predictions count equally. The sum adds up all $n$ absolute errors, and dividing by $n$ gives the average. The result is in the **data's original units** — thousands of employees for our series. If the MAE is 200, the model is off by about 200 thousand employees on average. Robust to occasional large errors, MAE is the workforce planner's default reporting metric.

**Root Mean Squared Error (RMSE).**

$$\text{RMSE} = \sqrt{\frac{1}{n}\sum_{t=1}^{n} \left( y_t - \hat{y}_t \right)^2}$$

Same forecast error $y_t - \hat{y}_t$, but now it is **squared** before averaging. Squaring makes large errors count disproportionately more than small ones — a single month off by 500 hurts far more than five months off by 100 each, even though the total absolute error is the same. The square root at the end brings the result back to approximately the data's units. Use RMSE when a large miss is disproportionately costly — stock-outs, surge planning, safety-critical capacity.

**Mean Absolute Percentage Error (MAPE).**

$$\text{MAPE} = \frac{100}{n}\sum_{t=1}^{n} \left| \frac{y_t - \hat{y}_t}{y_t} \right|$$

The error is divided by the actual value $y_t$ before averaging, which turns it into a **percentage**. The result is scale-free and comparable across series with different magnitudes — a MAPE of 3% means "the forecast is off by 3% of the actual, on average." This is the metric that speaks to non-technical audiences ("we are off by 3%"). Two caveats: MAPE breaks if any $y_t$ is near zero (division by near-zero inflates the percentage), and it is asymmetric — over-forecasts produce larger percentages than equally-sized under-forecasts.

**Mean Absolute Scaled Error (MASE).**

$$\text{MASE} = \frac{\text{MAE}}{\text{MAE}_{\text{seasonal-naive, in-sample}}}$$

The numerator is your model's MAE; the denominator is the MAE that the seasonal-naive baseline achieves on the training data. The ratio answers a simple question: **does your model beat the free baseline?** A MASE below 1 means yes — your model's errors are smaller than what you get for free from seasonal-naive. A MASE above 1 means no — the free baseline is better than your model, and you do not have a champion. MASE is scale-free and comparable across series with different units and magnitudes.

### 7.2 When to use which

| Use this metric ... | ... when |
|---|---|
| **MAE** | Reporting in business units (employees, dollars, units sold) and you want a robust default. |
| **RMSE** | Large misses cost much more than small ones (stock-outs, surge planning, safety-critical capacity). |
| **MAPE** | Reporting to non-technical audiences ("we are off by 3% on average") — and only when *y* is far from zero across the validation window. |
| **MASE** | Comparing forecasts across multiple series with different scales (cross-region demand, multi-product KPIs). |

**A question that often comes up here:** *"If the metrics rank models differently, which do I trust?"* The one whose error structure matches your business cost. If a stock-out costs 10× as much as overstock, RMSE is the honest metric — squaring penalizes the rare large miss exactly the way the cost matrix does. There is no "best" metric in the abstract; there is only the metric that aligns with consequences.


The four classical benchmarks are now one-liners using `sktime`'s `NaiveForecaster` — each `strategy` parameter maps to one of the manual functions you would have written by hand. **`strategy="mean"`** forecasts the historical average (a flat line). **`strategy="last"`** carries the last observed value forward (the Naive baseline). **`strategy="last"` with `sp=12`** replays the most recent 12 months (Seasonal-Naive). **`strategy="drift"`** extrapolates a straight line from the first training observation to the last (Drift). The code cell below fits each one on the training data and forecasts the validation window.

In [ ]:
# Four classical benchmarks using sktime's NaiveForecaster.
# Each strategy maps to the manual function we would have written by hand.
baselines = {
    "Mean":          NaiveForecaster(strategy="mean"),
    "Naive":         NaiveForecaster(strategy="last"),
    "Seasonal-Naive": NaiveForecaster(strategy="last", sp=12),
    "Drift":         NaiveForecaster(strategy="drift"),
}

# Fit each baseline on the training data and forecast the validation window
# For the visual demo, fit on the first 75% of the training window
# and forecast the last 25% — a single-shot preview before CV.
n_demo = int(len(y_train) * 0.75)
y_demo_fit = y_train.iloc[:n_demo]
y_demo_eval = y_train.iloc[n_demo:]
fh_demo = np.arange(1, len(y_demo_eval) + 1)
preds = {}
for name, model in baselines.items():
    model.fit(y_demo_fit)
    preds[name] = model.predict(fh=fh_demo).values

# Plot all four against the validation actuals
fig, ax = plt.subplots(figsize=(12, 5))
train_tail = y_demo_fit.iloc[-60:]
ax.plot(train_tail.index.to_timestamp(), train_tail.values, color="black", label="Fit window (last 60 mo)")
ax.plot(y_demo_eval.index.to_timestamp(), y_demo_eval.values, color="black", linestyle="--", label="Held-out (actual)")
for name, p in preds.items():
    ax.plot(y_demo_eval.index.to_timestamp(), p, label=name)
ax.set_title("Four classical benchmarks on the validation window")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

# --- Multi-metric evaluation utility ---
def all_metrics(y_true, y_pred, training_y, season=12):
    yt = np.asarray(y_true, dtype=float)
    yp = np.asarray(y_pred, dtype=float)
    mae = np.mean(np.abs(yt - yp))
    rmse = np.sqrt(np.mean((yt - yp) ** 2))
    mape = np.mean(np.abs((yt - yp) / yt)) * 100.0
    th = np.asarray(training_y, dtype=float)
    seasonal_naive_errors = np.abs(th[season:] - th[:-season])
    mae_naive = seasonal_naive_errors.mean()
    mase = mae / mae_naive if mae_naive > 0 else np.nan
    return {"MAE": mae, "RMSE": rmse, "MAPE": mape, "MASE": mase}

benchmark_table = pd.DataFrame({
    name: all_metrics(y_demo_eval.values, p, y_demo_fit.values)
    for name, p in preds.items()
}).T
print("Four classical benchmarks \u2014 single-shot validation, all four metrics:")
print(benchmark_table.round(3))

print("\nRanking under each metric (1 = best):")
print(benchmark_table.rank(axis=0).astype(int))

**Reading the output:**

Three outputs to read in sequence: the plot, the metric table, and the ranking table.

**The plot** shows five lines crossing the validation window: the four baseline forecasts and the black dashed actual values. Look for which colored line tracks the black actuals most closely. On US Retail Trade, **Seasonal-Naive** (which replays the last 12 months of training forward) typically hugs the actuals best — it captures the annual peaks and troughs that the other baselines miss. **Mean** is visibly the worst — it draws a flat horizontal line through the middle of the validation window, missing both the trend and the seasonality. **Naive** (carry the last training value forward) captures the level but misses the seasonal swing. **Drift** (straight-line extrapolation) captures the trend direction but also misses the seasonality.

**The metric table** puts numbers behind the visual impression. Four rows (one per baseline), four columns (MAE, RMSE, MAPE, MASE). Read the MAE column first — it is in the planner's native units (thousands of employees). Seasonal-Naive typically has the lowest MAE, confirming what the plot showed. The MASE column is the reality check: a MASE below 1.0 means the baseline beats the in-sample seasonal-naive benchmark; above 1.0 means it lost. By definition, Seasonal-Naive's MASE is near 1.0 (it *is* the reference baseline for that metric).

**The ranking table** shows where the four metrics agree and where they disagree. When all four metrics rank the same baseline first, the ranking is **robust** — you can state the winner with confidence. When rankings disagree (say, MAE picks Seasonal-Naive but RMSE picks Drift), the disagreement is the teaching moment: MAE treats every error equally, while RMSE punishes the occasional large miss more heavily. The choice between them is a **business** choice — does the workforce planner's procurement plan tolerate steady small misses (favor MAE) or is a single large miss catastrophic (favor RMSE)?

These rankings depend on the data's structural features. The next subsection makes that concrete by running the same benchmarks on a series with no seasonality at all.

---

### 7.3 Non-Seasonal Contrast — Google Daily Stock Prices

The US Retail Trade series above has clear annual seasonality, which is why **Seasonal-Naive** was such a strong baseline. But many business series — daily stock prices, intraday traffic, hourly server load, web-conversion rates — have little or no calendar seasonality. The classical benchmarks behave very differently there: **Drift** (linear extrapolation from start to end of training) often beats Seasonal-Naive by a lot because there is no annual pattern to lean on, and **Naive** (carry the last training value forward) can also be competitive because consecutive days are highly correlated.

To make the contrast concrete, we run the same benchmarks on Google daily closing prices: train on 2015, test on January 2016. The pedagogical point is that **the choice of benchmark depends on the data’s structural features, not on the model**.

In [ ]:
# Load Google daily closing prices from the GAFA stock dataset
GAFA_URL = (
    "https://raw.githubusercontent.com/davi-moreira/"
    "2026Summer_predictive_analytics_purdue_MGMT474/main/"
    "lecture_slides/08_time_series/data/gafa_stock.csv"
)
gafa = pd.read_csv(GAFA_URL, parse_dates=["ds"])
goog = (
    gafa[gafa["unique_id"] == "GOOG_Close"]
    .loc[:, ["ds", "y"]]
    .sort_values("ds")
    .reset_index(drop=True)
)

# Convert to sktime format. Stock data trades on irregular days
# (weekends and holidays are missing). PeriodIndex("D") gives each
# observation a daily period label without requiring a regular grid,
# so sktime can compute forecast horizons.
y_goog = goog.set_index("ds")["y"]

# Train: 2015 (full calendar year). Test: January 2016.
goog_train = y_goog["2015"]
goog_test  = y_goog["2016-01"]
goog_train.index = goog_train.index.to_period("D")
goog_test.index  = goog_test.index.to_period("D")

print(f"Train: {goog_train.index[0]} -> {goog_train.index[-1]}  (n={len(goog_train)})")
print(f"Test : {goog_test.index[0]} -> {goog_test.index[-1]}  (n={len(goog_test)})")

# Three classical benchmarks using NaiveForecaster
fh_g = np.arange(1, len(goog_test) + 1)
goog_baselines = {
    "Mean":  NaiveForecaster(strategy="mean"),
    "Naive": NaiveForecaster(strategy="last"),
    "Drift": NaiveForecaster(strategy="drift"),
}
preds_g = {}
for name, model in goog_baselines.items():
    model.fit(goog_train)
    preds_g[name] = model.predict(fh=fh_g).values

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(goog_train.index.to_timestamp(), goog_train.values,
        color="grey", alpha=0.6, linewidth=0.8, label="2015 (train)")
ax.plot(goog_test.index.to_timestamp(), goog_test.values,
        color="black", linewidth=1.5, label="Jan 2016 (test, actual)")
for name, p in preds_g.items():
    ax.plot(goog_test.index.to_timestamp(), p, label=name, linewidth=1.2)
ax.axvline(pd.Timestamp("2016-01-01"), color="red", linestyle=":", alpha=0.5,
           label="Train / test boundary")
ax.set_title("Google Daily Closing Price \u2014 2015 Train, Jan 2016 Test")
ax.set_xlabel("Date")
ax.set_ylabel("Closing price (USD)")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

goog_table = pd.DataFrame({
    name: all_metrics(goog_test.values, p, goog_train.values, season=1)
    for name, p in preds_g.items()
}).T
print("\nAccuracy table \u2014 Google daily, January 2016 horizon (MASE referenced to 1-step naive):")
print(goog_table.round(3))
print("\nRanking under each metric (1 = best):")
print(goog_table.rank(axis=0).astype(int))

**Reading the output:**

The ranking on Google daily prices typically **flips** relative to US Retail Trade — a concrete demonstration that the right baseline depends on the data, not on a modeling assumption.

**In the plot**, the 2015 training data appears in grey and the January 2016 test window in black. The three forecast lines extend past the red vertical cutoff. **Drift** (the straight-line extrapolation from the first to the last training price) typically tracks the January actuals most closely — it captures the mild upward or downward momentum of the 2015 price trajectory. **Naive** (carry the last December 2015 closing price forward) draws a flat line that is competitive because consecutive trading days are highly correlated; tomorrow's price is almost always close to today's. **Mean** (the average of all 2015 closing prices) is usually the worst — it projects a price from mid-2015 into January 2016, throwing away the level the stock actually closed at.

**Why the flip?** US Retail Trade has strong annual seasonality, so Seasonal-Naive — which replays last year's pattern — is the natural winner. Google daily stock has no calendar seasonality (there is no "December peak" in closing prices), so Seasonal-Naive is not even in the race. The dominant structure is a slow random-walk-like drift, which is exactly what the Drift baseline captures.

**MASE with season = 1** deserves a note. For the retail employment series, MASE used the 12-step seasonal-naive baseline as the denominator. For a non-seasonal series, the natural denominator is the 1-step naive baseline (a random walk). The formula is the same; only the reference baseline changes. A MASE below 1 still means "the model beats the free baseline" — the baseline is just a different one.

The business takeaway: **build the classical benchmarks first on every new series; let the data tell you which one your learned model has to beat.** A model evaluated against the wrong baseline can look impressive while adding no real value.

With the benchmarks established on both seasonal and non-seasonal series, section 8 asks: can a simple learned model — linear regression on lag features — beat those free baselines?

---

## 8. Cross-Validated Comparison — Five Candidates, Identical Folds

The baselines in §7 gave us a visual preview, but a single-split evaluation is one roll of the dice. We now compare all five forecasters on the **same** walk-forward folds from §6.

Three of the five candidates are the `NaiveForecaster` baselines from §7. The other two are **learned models** that use the ACF-driven feature selection from §3.5: a `LinearRegression` and a `Ridge`, each fitted on exactly **lag-1 and lag-12** — the two lags the ACF identified as carrying the strongest signal.

To build these models, we combine two tools from earlier in the course. First, `make_reduction` from `sktime` constructs a 12-column lag matrix inside its `.fit()` call (one column per lag, from `lag1` through `lag12`). Second, a sklearn `Pipeline` — the same pattern nb02 taught — selects only columns 0 and 11 (lag-1 and lag-12) before feeding them to the regressor. The result is a forecaster that:

- uses exactly the two features the ACF justified,
- constructs them inside `.fit()` so they respect fold boundaries during CV (no leakage by construction), and
- handles recursive multi-step forecasting automatically (`strategy="recursive"` feeds predictions back as lag inputs).

This is the **Pipeline principle from nb02** applied to forecasting: feature engineering inside the model, not outside it.

In [ ]:
# ACF-driven lag selection: pick only lag-1 and lag-12 from the
# 12-column lag matrix that make_reduction constructs.
def select_lag1_lag12(X):
    """Pick lag-1 (column 0) and lag-12 (column 11)."""
    return X[:, [0, 11]]

# Build sklearn Pipelines (same pattern as nb02):
# Step 1: select the two ACF-justified lags
# Step 2: fit the regressor on those two features only
lr_pipeline = Pipeline([
    ("select_lags", FunctionTransformer(select_lag1_lag12)),
    ("regressor", LinearRegression()),
])
ridge_pipeline = Pipeline([
    ("select_lags", FunctionTransformer(select_lag1_lag12)),
    ("regressor", Ridge(alpha=1.0, random_state=RANDOM_SEED)),
])

# Five-candidate walk-forward comparison on identical folds.
candidates = {
    "Mean":               NaiveForecaster(strategy="mean"),
    "Naive":              NaiveForecaster(strategy="last"),
    "Seasonal-Naive":     NaiveForecaster(strategy="last", sp=12),
    "Linear [lag1,lag12]": make_reduction(lr_pipeline, window_length=12,
                                          strategy="recursive"),
    "Ridge  [lag1,lag12]": make_reduction(ridge_pipeline, window_length=12,
                                          strategy="recursive"),
}

# --- Walk-forward CV loop ---
results = pd.DataFrame()
for name, forecaster in candidates.items():
    fold_maes = []
    for tr_idx, va_idx in cv.split(y_train):
        y_cv_train = y_train.iloc[tr_idx]
        y_cv_val   = y_train.iloc[va_idx]
        fc = forecaster.clone()
        fc.fit(y_cv_train)
        y_cv_pred = fc.predict(fh=np.arange(1, len(y_cv_val) + 1))
        fold_maes.append(mean_absolute_error(y_cv_val, y_cv_pred))
    results[name] = np.array(fold_maes)

# --- Selection metric (MAE) with Student's t 95% CI ---
t_crit = student_t.ppf(0.975, df=n_folds - 1)
summary = pd.DataFrame({
    "MAE_mean": results.mean(),
    "MAE_sd":   results.std(ddof=1),
    "CI_halfwidth": results.std(ddof=1) / np.sqrt(n_folds) * t_crit,
}).sort_values("MAE_mean")
summary["CI_low"] = summary["MAE_mean"] - summary["CI_halfwidth"]
summary["CI_high"] = summary["MAE_mean"] + summary["CI_halfwidth"]
print("Selection metric (MAE) \u2014 walk-forward CV with 95% CI:")
print(summary.round(2))

# --- Multi-metric sensitivity check ---
def per_fold_all_metrics(forecaster, y_data, cv_splitter, season=12):
    out = {"MAE": [], "RMSE": [], "MAPE": [], "MASE": []}
    for tr_idx, va_idx in cv_splitter.split(y_data):
        y_tr = y_data.iloc[tr_idx]
        y_va = y_data.iloc[va_idx]
        fc = forecaster.clone()
        fc.fit(y_tr)
        yp = fc.predict(fh=np.arange(1, len(y_va) + 1)).values
        yt = y_va.values
        out["MAE"].append(np.mean(np.abs(yt - yp)))
        out["RMSE"].append(np.sqrt(np.mean((yt - yp) ** 2)))
        out["MAPE"].append(np.mean(np.abs((yt - yp) / yt)) * 100.0)
        th = y_tr.values
        if len(th) > season:
            mae_naive = np.mean(np.abs(th[season:] - th[:-season]))
            out["MASE"].append(np.mean(np.abs(yt - yp)) / mae_naive)
        else:
            out["MASE"].append(np.nan)
    return {k: np.array(v) for k, v in out.items()}

all_results = {name: per_fold_all_metrics(fc, y_train, cv)
               for name, fc in candidates.items()}
metric_means = pd.DataFrame({m: {name: r[m].mean() for name, r in all_results.items()}
                             for m in ["MAE", "RMSE", "MAPE", "MASE"]})
metric_means = metric_means.loc[summary.index]
print("\nMulti-metric sensitivity check \u2014 per-candidate mean:")
print(metric_means.round(3))
print("\nRanking under each metric (1 = best):")
print(metric_means.rank(axis=0).astype(int))

# --- Selection bar chart ---
fig, ax = plt.subplots(figsize=(11, 5))
y_pos = np.arange(len(summary))
ax.barh(y_pos, summary["MAE_mean"],
        xerr=summary["CI_halfwidth"], color="#1f77b4", edgecolor="black", capsize=4)
ax.set_yticks(y_pos)
ax.set_yticklabels(summary.index)
ax.invert_yaxis()
ax.set_xlabel("MAE (walk-forward CV; bars = 95% CI)")
ax.set_title("Five-candidate forecast comparison \u2014 selection metric (MAE)")
plt.tight_layout()
plt.show()

# --- Champion's coefficients (2 values: lag1 and lag12) ---
champ_fc = make_reduction(lr_pipeline, window_length=12, strategy="recursive")
champ_fc.fit(y_train)
coefs = champ_fc.estimator_.named_steps["regressor"].coef_
intercept = champ_fc.estimator_.named_steps["regressor"].intercept_
print(f"\nChampion coefficients:  lag1 = {coefs[0]:.3f},  lag12 = {coefs[1]:.3f}")
print(f"Intercept: {intercept:.1f}")
print("The ACF predicted lag-1 and lag-12 would dominate — these two")
print("coefficients ARE the entire model.")

**Reading the output:**

Three outputs, read in sequence.

**The MAE selection table** is the primary decision tool. Each row is one candidate model; the columns show the mean MAE across five walk-forward folds, the standard deviation, the CI half-width, and the lower and upper bounds of the 95% CI. The candidates are sorted by mean MAE — the top row is the current leader. Look at the CI columns: if the leader's CI does not overlap with the runner-up's CI, the leader is **genuinely better** on this metric. If the CIs overlap, you cannot distinguish them statistically — pick the simpler model.

**The multi-metric sensitivity table** shows the mean score for each candidate under all four metrics (MAE, RMSE, MAPE, MASE). The ranking table below it asks *"would I pick a different champion if I cared about a different metric?"* If all four metrics rank the same model first, the champion is **robust** — ship it with confidence. If the rankings disagree (model A wins on MAE but model B wins on RMSE), the disagreement points you to the metric whose error structure matches the business cost.

**The bar chart** makes the CI comparison visual. Horizontal bars show each candidate's mean MAE; error bars show the 95% CI. Look for gaps between bars: a clear gap with no error-bar overlap means a genuine difference; overlapping error bars mean statistically indistinguishable candidates.



**The champion's coefficients** confirm the ACF analysis: the model has exactly two weights — one for lag-1 (short-term momentum) and one for lag-12 (annual seasonality). Together with the intercept, these three numbers ARE the entire model. The workforce planner can read them directly: "each additional thousand employees last month contributes about X hundred to next month's forecast; each additional thousand from the same month last year contributes about Y hundred." No black box — any analyst can verify the forecast by hand.

Three interpretation rules borrowed from nb08:

1. **Non-overlapping CIs** between candidate A and candidate B → A is genuinely better on the selection metric.
2. **Overlapping CIs** → no statistical evidence to prefer one over the other; pick the simpler model (Occam's razor).
3. **Mean is far worse than the rest** → expected. It ignores trend and seasonality entirely; it is only here as a sanity floor.

If the linear and Ridge models have overlapping CIs, **Ridge does not earn its place** here — the regularization adds machinery without a measurable payoff. That is the right outcome for a 2-feature model; Ridge typically wins when the feature count is large and multicollinearity is a real risk.

---

## 📝 PAUSE-AND-DO Exercise 1 — Compare [lag1, lag12] vs all 12 lags (10 minutes)

**Task:** The ACF-driven model uses only lag-1 and lag-12. But `make_reduction(LinearRegression(), window_length=12)` without the lag-selection Pipeline uses *all 12 lags*. Does the full 12-lag model earn its place over the 2-lag model by **non-overlapping CIs**?

**Hints:**
- Build `make_reduction(LinearRegression(), window_length=12, strategy="recursive")` — no Pipeline, no lag selection.
- Run it through the same CV folds and add its fold MAEs to the results DataFrame as `"Linear [all 12 lags]"`.
- Rebuild the summary table with all six candidates and compare CIs.
- Overlap = the extra lags did not earn their place; the ACF-driven 2-feature model is sufficient.

Type your code in the cell below.

> 💡 **Gemini Prompt:** *"I have a walk-forward CV comparison of five forecasters on monthly US retail employment data using sktime's ExpandingWindowSplitter (cv). The candidates include NaiveForecaster baselines and make_reduction with a Pipeline that selects only lag-1 and lag-12 from a 12-lag matrix. I want to add a sixth candidate: make_reduction(LinearRegression(), window_length=12, strategy='recursive') — a model using ALL 12 lags without the lag-selection step. Run it through the same CV folds using .clone() / .fit() / .predict(), add its per-fold MAEs to the results DataFrame as 'Linear [all 12 lags]', rebuild the summary table with Student's t 95% CIs (t_crit and n_folds are already defined), and print the updated table plus a horizontal bar chart comparing all six candidates."*
>
> **After running, verify:**
> - [ ] The new model `Linear [all 12 lags]` appears in the summary table alongside the original five candidates
> - [ ] The CI for the 12-lag model overlaps (or does not overlap) with `Linear [lag1,lag12]` — note which
> - [ ] The bar chart shows error bars for all six candidates
> - [ ] No test-set data was used anywhere

In [ ]:
# YOUR SOLUTION CODE HERE

# Hints:
# lr_all12 = make_reduction(LinearRegression(), window_length=12, strategy="recursive")
# fold_maes_12 = []
# for tr_idx, va_idx in cv.split(y_train):
#     fc = lr_all12.clone()
#     fc.fit(y_train.iloc[tr_idx])
#     pred = fc.predict(fh=np.arange(1, len(va_idx) + 1))
#     fold_maes_12.append(mean_absolute_error(y_train.iloc[va_idx], pred))
# results_ex1 = results.copy()
# results_ex1["Linear [all 12 lags]"] = np.array(fold_maes_12)
# Build the updated summary table, compare CIs.

## 9. Opening the Locked Test Window — One-Shot Evaluation

We now do the time-series analog of nb14's "test-set opening ceremony." The ritual is the same: pick the champion, refit on the full training window, predict the locked test window once, and read the verdict — **INSIDE / ABOVE / BELOW** the CV 95% CI. Because we used an 80/20 split (no separate validation set), the champion is already trained on the most recent pre-test data — no concatenation step needed. What is new here is the **prediction interval**: we estimate the residual sigma from walk-forward folds (not from the final training fit) and wrap a 95% Gaussian band around each point forecast. The workforce planner gets not just "forecast = X" but "forecast = X ± Y with Z% empirical coverage" — a deliverable the legislature can read.

In [ ]:
# Step 1: collect walk-forward residuals for PI sigma estimation.
fold_residuals = []
for tr_idx, va_idx in cv.split(y_train):
    y_cv_train = y_train.iloc[tr_idx]
    y_cv_val   = y_train.iloc[va_idx]
    fc = make_reduction(lr_pipeline, window_length=12, strategy="recursive")
    fc.fit(y_cv_train)
    pred_fold = fc.predict(fh=np.arange(1, len(y_cv_val) + 1))
    fold_residuals.extend(y_cv_val.values - pred_fold.values)
fold_residuals = np.array(fold_residuals)
sigma_residual = fold_residuals.std(ddof=1)
print(f"Walk-forward residual sigma: {sigma_residual:.2f} (units: thousands of employees)")

# Step 2: refit champion on the full training window
champion = make_reduction(lr_pipeline, window_length=12, strategy="recursive")
champion.fit(y_train)

# Step 3: point forecast on the locked test window
fh_test = np.arange(1, len(y_test) + 1)
y_test_pred = champion.predict(fh=fh_test)
test_mae = mean_absolute_error(y_test, y_test_pred)

# Step 4: 95% prediction interval (Gaussian assumption on walk-forward residuals)
z_95 = 1.96
y_test_lower = y_test_pred.values - z_95 * sigma_residual
y_test_upper = y_test_pred.values + z_95 * sigma_residual

# Step 5: empirical coverage
inside = ((y_test.values >= y_test_lower) & (y_test.values <= y_test_upper)).mean()

# Verdict
champ_row = summary.loc["Linear [lag1,lag12]"]
cv_low, cv_high = champ_row["CI_low"], champ_row["CI_high"]
verdict = ("INSIDE the CV 95% CI" if cv_low <= test_mae <= cv_high
           else "ABOVE the CV 95% CI (overfitting?)" if test_mae > cv_high
           else "BELOW the CV 95% CI (lucky test window?)")

print(f"\nChampion: Linear [lag1,lag12]")
print(f"CV MAE 95% CI : [{cv_low:.2f}, {cv_high:.2f}]")
print(f"Test MAE      : {test_mae:.2f}  ->  {verdict}")
print(f"Empirical 95% PI coverage on test: {inside*100:.1f}%  (nominal: 95.0%)")

# Plot
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(y_train.index.to_timestamp(), y_train.values, color="#1f77b4",
        label="Train", linewidth=0.8)
ax.plot(y_test.index.to_timestamp(), y_test.values, color="#d62728",
        label="Test (actual)", linewidth=1.5)
ax.plot(y_test.index.to_timestamp(), y_test_pred.values, color="#2ca02c",
        linestyle="--", label="Champion forecast", linewidth=1.5)
ax.fill_between(y_test.index.to_timestamp(), y_test_lower, y_test_upper,
                color="#2ca02c", alpha=0.20, label="95% prediction interval")
ax.set_title("Locked Test Window \u2014 Forecast + 95% Prediction Interval")
ax.legend()
plt.tight_layout()
plt.show()

**Reading the output:**

Three outputs, one verdict.

**The printed summary** shows the champion name, the CV MAE 95% CI (from §9), the test MAE, and the verdict. The verdict follows nb14's protocol:

- **INSIDE the CV 95% CI** means the walk-forward CV estimate generalized to the truly unseen test window. The CV-based selection was honest — the workforce planner can quote the test MAE with confidence.
- **ABOVE the CV 95% CI** signals that the champion performed worse on the test window than the CV predicted — possible overfitting to the training history, or a structural break in the test period (a recession the model could not foresee).
- **BELOW the CV 95% CI** means the champion performed better than expected — the test window was easier than the average CV fold. Unusual but not alarming.

**The empirical coverage** is the new diagnostic. We constructed the 95% PI under a **Gaussian assumption** on the walk-forward residuals: $\hat{y}_t \pm 1.96 \cdot \sigma_{\text{residual}}$. Then we asked *"what fraction of locked-test actuals actually fell inside that interval?"*. Three reading rules:

- **Coverage near 95%** (say, 90–98%): the Gaussian assumption holds, the interval is honest, and the workforce planner can quote it to the legislature.
- **Coverage well below 95%** (say, 70–85%): the model is **overconfident** — the residuals have heavier tails than Gaussian, or there are structural breaks the lag-feature model cannot capture (recessions, policy shocks). The interval needs to be widened before reporting.
- **Coverage near 100%**: the interval is **too wide** — typically because the residual sigma was inflated by a few outlier folds. Tighter intervals would still cover the right amount and be more useful to a decision-maker.

**The plot** shows the full series in three colors (blue train, orange val, red test actuals) plus the green dashed champion forecast and a shaded green band for the 95% prediction interval. Look for whether the red test actuals stay inside the green band — that is the coverage check visualized. Any point where the red line escapes the band is a moment the model's uncertainty estimate was too narrow.

For the workforce planner, the deliverable is no longer just *"forecast = X"* but *"forecast = X, 95% interval [X\u2212Y, X+Y], and the model has been cross-validated to deliver that coverage on held-out data."* That is the line that goes on the M4 poster.

> **A question that often comes up here:** *"Why use the residual sigma from CV folds instead of from the final training fit?"* Because the final training residuals are in-sample — the model fitted those rows. Walk-forward residuals are out-of-sample; they reflect the noise the model will encounter on truly future data. Using in-sample residuals would systematically underestimate sigma and produce overconfident intervals. Same principle as nb08's CV CIs.

---

> 💡 **Gemini Prompt:** *"I have a walk-forward CV setup using sktime's ExpandingWindowSplitter on monthly US retail employment data. The champion is make_reduction(LinearRegression(), window_length=12, strategy='recursive'). Sweep Ridge alpha over [0.01, 0.1, 1, 10, 100] — at each alpha, create make_reduction(Ridge(alpha=alpha, random_state=RANDOM_SEED), window_length=12, strategy='recursive'), run it through the CV folds using .clone()/.fit()/.predict(), and collect the mean MAE and CI half-width. Build a DataFrame, print it, plot alpha on a log x-axis vs MAE with error bars, and compare the best Ridge CI to the Linear CI from the summary table."*
>
> **After running, verify:**
> - [ ] The table shows five rows (one per alpha) with MAE_mean and CI half-width columns
> - [ ] The plot has alpha on a log-scaled x-axis with error bars at each point
> - [ ] A printed comparison states whether the best Ridge CI overlaps with the Linear CI
> - [ ] All evaluation uses walk-forward CV on training data only — no test-set leak

> 💡 **Gemini Prompt:** *"I have a walk-forward CV setup using sktime's ExpandingWindowSplitter on monthly US retail employment data with lag1 and lag12 features in df_lag_train. The helper I want to sweep Ridge alpha over [0.01, 0.1, 1, 10, 100]. At each alpha, create make_reduction(Ridge(alpha=alpha, random_state=RANDOM_SEED), window_length=12, strategy='recursive'), run it through the CV folds using .clone()/.fit()/.predict(), and collect the mean MAE and CI half-width (t_crit and n_folds are already defined). Build a DataFrame, print it, plot alpha on a log x-axis vs MAE with error bars, and compare the best Ridge CI to the Linear CI from the summary table."*
>
> **After running, verify:**
> - [ ] The table shows five rows (one per alpha) with MAE_mean and CI half-width columns
> - [ ] The plot has alpha on a log-scaled x-axis with error bars at each point
> - [ ] A printed comparison states whether the best Ridge CI overlaps with the Linear CI
> - [ ] All evaluation uses walk-forward CV on training data only — no test-set leak

In [ ]:
# YOUR SOLUTION CODE HERE

# Hints:
# alphas = [0.01, 0.1, 1, 10, 100]
# rows_ridge = []
# for a in alphas:
#     ridge_fc = make_reduction(
#         Pipeline([("select_lags", FunctionTransformer(select_lag1_lag12)),
#                   ("regressor", Ridge(alpha=a, random_state=RANDOM_SEED))]),
#         window_length=12, strategy="recursive")
#     fold_maes = []
#     for tr_idx, va_idx in cv.split(y_train):
#         fc = ridge_fc.clone()
#         fc.fit(y_train.iloc[tr_idx])
#         pred = fc.predict(fh=np.arange(1, len(va_idx) + 1))
#         fold_maes.append(mean_absolute_error(y_train.iloc[va_idx], pred))
#     rows_ridge.append({"alpha": a, "MAE_mean": np.mean(fold_maes),
#                         "CI_hw": np.std(fold_maes, ddof=1)/np.sqrt(n_folds)*t_crit})
# Then plot with errorbar() on a log-x axis.

## 10. Forecast Accuracy Diagnostics — Residuals and Horizon

Two questions are worth answering before we wrap up. First: **why did we estimate the prediction-interval sigma from walk-forward residuals instead of in-sample training residuals?** A side-by-side comparison answers it visually. Second: **does forecast error grow as we predict further ahead?** A rolling-forecast-origin sweep across horizons answers that one directly.

These two diagnostics complete the toolkit a workforce planner needs to defend a forecast: the **point forecast** (§7-9), the **prediction interval** (§9), the **residual diagnostic** (§10.1), and the **horizon curve** (§10.2).


### 10.1 In-sample residuals vs walk-forward residuals

If the champion is fit on training data and we measure its residuals on those *same* rows, we get **in-sample** residuals — the errors from data the model has already seen and optimized against. Those residuals are systematically smaller than residuals on truly unseen rows, because the model has "memorized" some of the training noise. The walk-forward residuals, by contrast, come from validation windows the model never saw during fitting — they reflect what the model will actually face on future data. The gap between the two is exactly the gap a prediction interval has to honor. If the workforce planner uses in-sample sigma to build a 95% PI, the interval will be too narrow and more than 5% of future observations will fall outside. That is why §9's PI used walk-forward residual sigma.

In [ ]:
# In-sample residuals: build the lag-feature matrix manually so we can
# compute in-sample predictions. make_reduction does not support
# in-sample prediction, but the underlying sklearn model is a plain
# LinearRegression — we can replicate what make_reduction does internally.
lag_matrix = pd.DataFrame(
    {f"lag{k}": y_train.shift(k) for k in [1, 12]}
).dropna()
y_aligned = y_train.loc[lag_matrix.index]
lr_insample = LinearRegression().fit(lag_matrix, y_aligned)
in_sample_pred = lr_insample.predict(lag_matrix)
in_sample_residuals = y_aligned.values - in_sample_pred
in_sample_mae = float(np.mean(np.abs(in_sample_residuals)))
in_sample_sigma = float(np.std(in_sample_residuals, ddof=1))

# Walk-forward residuals already computed in \u00a79 as `fold_residuals`
cv_mae = float(np.mean(np.abs(fold_residuals)))
cv_sigma = float(sigma_residual)

cmp_table = pd.DataFrame({
    "MAE":             [in_sample_mae, cv_mae],
    "Residual sigma":  [in_sample_sigma, cv_sigma],
}, index=["In-sample (training fit)", "Walk-forward CV (out-of-sample)"])
print("Residual diagnostics \u2014 same champion, two ways of measuring its noise:")
print(cmp_table.round(2))
print(f"\nRatio (CV / in-sample) MAE   : {cv_mae / in_sample_mae:.2f}x")
print(f"Ratio (CV / in-sample) sigma : {cv_sigma / in_sample_sigma:.2f}x")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(in_sample_residuals, bins=40, color="#1f77b4", alpha=0.6, label="In-sample")
axes[0].hist(fold_residuals, bins=40, color="#d62728", alpha=0.6, label="Walk-forward CV")
axes[0].axvline(0, color="black", linewidth=0.5)
axes[0].set_xlabel("Residual (units: thousands of employees)")
axes[0].set_ylabel("Count")
axes[0].set_title("Residual distributions")
axes[0].legend()

axes[1].bar(["In-sample\n(training fit)", "Walk-forward CV\n(out-of-sample)"],
            [in_sample_mae, cv_mae],
            color=["#1f77b4", "#d62728"], edgecolor="black")
axes[1].set_ylabel("Mean absolute residual")
axes[1].set_title("Residual MAE \u2014 why the PI used CV, not in-sample")
plt.tight_layout()
plt.show()

**Reading the output:**

Two panels side by side make the case visually.

**Left panel (Residual histograms).** The blue histogram shows in-sample residuals — the errors from the champion fitted on training data and evaluated on those same training rows. The red histogram shows walk-forward CV residuals — the errors from validation windows the model never saw during fitting. The red histogram is **wider and more dispersed** than the blue one. The in-sample residuals cluster tightly around zero because the model optimized against those exact rows; the walk-forward residuals spread further because the model faces unseen dynamics — seasonal shifts it did not train on, trend changes it could not anticipate.

**Right panel (MAE bar chart).** Two bars compare the mean absolute residual from each source. The walk-forward CV MAE is typically **1.5\u00d7 to 3\u00d7 larger** than the in-sample MAE on a series with strong autocorrelation like this one. That multiplier is the honesty gap — the difference between how good the model *looks* on data it has seen and how good it *actually is* on data it has not.

**Why this matters for the workforce planner:** §9's prediction interval used the walk-forward sigma (the red histogram's spread), not the in-sample sigma (the blue histogram's spread). If we had used the in-sample sigma, the 95% prediction interval would have been **systematically too narrow** — claiming 95% coverage on paper while letting more than 5% of true future values fall outside the band. The workforce planner would have quoted a tight interval to the legislature, and then been embarrassed when actual employment repeatedly landed outside it.

This is the time-series version of the lesson nb08 taught for k-fold: **in-sample evaluation is overconfident; out-of-sample evaluation is honest.** The same principle applies to point estimates (MAE) and to uncertainty estimates (sigma).

### 10.2 Forecast horizon and accuracy

A workforce planner who asks *"what will retail employment be next month?"* gets a tight answer. A planner who asks *"what about next year?"* gets a much wider band — forecast error compounds with distance. To quantify this, we sweep horizon $h \in \{1, 2, \dots, 12\}$ using a **rolling forecast origin** (each `ExpandingWindowSplitter` fold supplies one cutoff) combined with **recursive multi-step forecasting**: the model predicts month 1, feeds that prediction back as `lag1` to predict month 2, feeds month 2 back to predict month 3, and so on. Each step uses the model's own imperfect output rather than actual data, so errors accumulate.

In [ ]:
# Sweep horizon h = 1..12 using make_reduction's recursive strategy.
# For each CV fold, fit the champion and predict h steps ahead.
HORIZONS = list(range(1, 13))
errors_by_h = {h: [] for h in HORIZONS}

for tr_idx, va_idx in cv.split(y_train):
    y_cv_train = y_train.iloc[tr_idx]
    y_cv_val   = y_train.iloc[va_idx]
    fc = make_reduction(lr_pipeline, window_length=12, strategy="recursive")
    fc.fit(y_cv_train)
    h_max = min(12, len(y_cv_val))
    preds_h = fc.predict(fh=np.arange(1, h_max + 1))
    actuals_h = y_cv_val.iloc[:h_max]
    for h in range(1, h_max + 1):
        errors_by_h[h].append((actuals_h.iloc[h-1] - preds_h.iloc[h-1]) ** 2)

rmse_by_h = {h: float(np.sqrt(np.mean(es))) for h, es in errors_by_h.items() if es}

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(list(rmse_by_h.keys()), list(rmse_by_h.values()),
        "o-", color="#9467bd", linewidth=2, markersize=8)
ax.set_xlabel("Forecast horizon h (months ahead)")
ax.set_ylabel("RMSE (recursive forecast)")
ax.set_title("Forecast accuracy degrades with horizon \u2014 Linear [lag1,lag12]")
ax.grid(alpha=0.3)
ax.set_xticks(list(rmse_by_h.keys()))
plt.tight_layout()
plt.show()

rmse_table = pd.Series(rmse_by_h, name="RMSE").round(2).to_frame()
rmse_table.index.name = "h (months ahead)"
print(rmse_table)

**Reading the output:**

The plot shows a single line: RMSE on the y-axis, forecast horizon (months ahead) on the x-axis, from 1 to 12.

**The line rises monotonically.** The 1-month-ahead RMSE is the smallest — the model's prediction for next month is its most accurate. By the time you reach 12 months ahead, the RMSE is typically **3\u00d7 to 5\u00d7 larger** than the 1-month value. The table below the plot prints the exact RMSE at each horizon so the workforce planner can quote specific numbers: "our 1-month forecast is accurate to within \u00b1X thousand employees; our 12-month forecast is accurate to within \u00b1Y thousand."

**Two effects compound to produce this degradation.** **Recursive feedback** is the first: for horizons beyond one month, the model uses its own imperfect predictions as `lag1` inputs. The month-1 prediction feeds into the month-2 prediction, the month-2 prediction feeds into month-3, and so on. Each step's error becomes part of the next step's input, so small mistakes at early horizons accumulate into larger mistakes at later horizons. **Information staleness** is the second: the model has access to actual lag values up to the forecast origin, but no information about events after that point. The further out it forecasts, the more events it cannot know about — recessions, policy shocks, structural changes in the retail labor market.

**Practical implication for the workforce planner:** the forecast quoted **one month out** can carry a tight prediction interval; the forecast quoted **one year out** cannot. When the legislature asks for a 12-month forecast, the planner should present it alongside this horizon-vs-RMSE curve and explicitly note the widening uncertainty. This curve is a strong candidate figure for the M4 poster's "Limitations" section — it demonstrates intellectual honesty about what the model can and cannot do.

> **A question that often comes up here:** *"Why does this differ from the \u00a79 CV table?"* The \u00a79 table averages across all rows in each validation fold, so it implicitly averages over many horizons mixed together. \u00a711.2 separates them — one RMSE per horizon — which is what you actually need when the business question is *"how far ahead can we trust this forecast?"*

With the point forecast, the prediction interval, the residual diagnostic, and the horizon curve all in hand, you have the full toolkit the workforce planner needs to defend a forecast. Section 11 pulls it all together.

---

## 11. Wrap-Up — Key Takeaways

1. **Forecasting is supervised learning with one structural rule: never let the future leak into the past.** That single rule changes the train/test split (recent slice held out), the cross-validation strategy (`ExpandingWindowSplitter` from `sktime`), and what counts as a feature (lags, not random shuffling).
2. **The Week-1 analytics workflow ports cleanly to time series.** EDA → split → baselines → linear features → regularization is the same recipe; only the partition strategy and feature engineering change.
3. **Naive baselines are surprisingly hard to beat.** If your fancy model does not beat seasonal-naive on identical CV folds with non-overlapping CIs, you do not have a champion — you have noise.
4. **The cost of lag features is the loss of the earliest rows.** A 12-month seasonal lag costs you the first year of history. Plan for it.
5. **Walk-forward CV is the time-series spine of CV-first evaluation,** exactly like `StratifiedKFold` was the classification spine in nb08–nb14.

### Beyond This Introduction

This notebook is an introduction — enough to build, evaluate, and defend a lag-feature linear forecast on a real business series. A dedicated time-series forecasting course covers substantially more ground:

| Topic | What it adds |
|---|---|
| **ARIMA / SARIMA** | The classical Box-Jenkins approach: differencing for stationarity, ACF/PACF-based order selection, seasonal terms. Still the benchmark in many industries. |
| **Exponential Smoothing (ETS)** | Holt-Winters and state-space models that weight recent observations more heavily than distant ones. The go-to for short-term inventory and demand planning. |
| **Prophet** | Meta's decomposable model with built-in holiday effects and automatic changepoint detection. Popular in e-commerce and retail forecasting. |
| **Multiple Seasonalities** | Series with daily, weekly, *and* annual cycles simultaneously — hourly electricity demand, web traffic, call-center staffing. |
| **Multivariate Forecasting (VAR)** | Using multiple related series (employment, GDP, consumer confidence) to forecast each other. Includes Granger causality — testing whether one series actually *predicts* another. |
| **Deep Learning (RNN / LSTM / Transformer)** | Sequence models that learn non-linear temporal dependencies from very long histories. Practical when you have thousands of series and large compute budgets. |
| **Hierarchical Reconciliation** | Forecasting at store, region, and national level simultaneously and reconciling the numbers to be consistent. Essential for retail chains and government agencies. |
| **Conformal Prediction Intervals** | Distribution-free uncertainty bands that guarantee coverage *without* the Gaussian assumption we used in §9. The fix when your residuals have heavy tails. |

The lag-feature regression you built today is not a toy — it is genuinely competitive on monthly business series with moderate trend and seasonality. The tools above extend the toolkit when the series is longer, more complex, or demands richer uncertainty quantification.

> **A question that often comes up here:** *"Where do RNNs and transformers fit?"* They are alternatives to lag-feature linear models when (a) the series is long enough (thousands of points, not 960), (b) the dependence is highly non-linear, and (c) you can spare an order of magnitude more compute. For business problems with a few decades of monthly history, a well-engineered lag-feature linear regression is almost always the right starting point — and often the right ending point. Deep learning gets the awareness module it deserves in **nb19**.

**Next stop — nb17: Data Communication and Poster Design.** Now that you have a forecast, a defensible CV-based comparison, and a clean test-set ceremony verdict, the question becomes how to **communicate** them: the six principles of data communication, the eleven-section poster architecture for the M4 deliverable, and the data-ink-ratio cleanup that turns a notebook plot into a poster figure.

---

## Participation Assignment Submission Instructions

1. **Complete both PAUSE-AND-DO exercises** (sections after 8 and 9).
2. **Run all cells** (`Runtime → Run all`).
3. **Save with output** (`File → Download → Download .ipynb`).
4. **Submit to Brightspace** as `nb16_time_series_forecasting_<your_lastname>.ipynb`.

**Bibliography**
- Hyndman & Athanasopoulos: *Forecasting: Principles and Practice* (FPP3) — the [free online textbook](https://otexts.com/fpp3/) is the deep dive on every concept above.
- sktime User Guide: `ExpandingWindowSplitter`, `NaiveForecaster`, and `make_reduction`.
- statsmodels: `STL` decomposition and the autocorrelation function.

<center>

# Thank you!

</center>
